In [1]:
import numpy as np
import pandas as pd
import json

In [2]:
df = pd.read_csv('./ieee-fraud-detection/train_transaction.csv')
display(df)

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df['TransactionDay'] = np.floor(df['TransactionDT'] / (24 * 60 * 60))

# 2. Calculate the Account Start Date 
df['Account_Start_Day'] = df['TransactionDay'] - df['D1']

# Convert Account_Start_Day to a nullable integer ('Int64').
# This drops the '.0' (e.g., -60.0 becomes -60) without crashing on NaNs.
df['Account_Start_Day'] = df['Account_Start_Day'].astype('Int64')

# 3. Create the UID by concatenating the defining features
uid_columns = ['card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'Account_Start_Day']

# Convert columns to string and clean up the NaN representations
for col in uid_columns:
    # astype(str) turns Int64 missing values into '<NA>' and float missing values into 'nan'.
    # We replace both with a standard 'NaN' so your UIDs look perfectly consistent.
    df[col] = df[col].astype(str).replace({'<NA>': 'NaN', 'nan': 'NaN'})

# Concatenate into a single UID string separated by underscores
df['UID'] = df[uid_columns].agg('_'.join, axis=1)

# 4. Sort the dataframe to create the chronological sequence
df_sequenced = df.sort_values(by=['UID', 'TransactionDT']).reset_index(drop=True)

print(df_sequenced[['TransactionDT', 'D1', 'Account_Start_Day', 'UID']].head())

   TransactionDT    D1 Account_Start_Day  \
0        4050851  83.0               -37   
1        8421815   7.0                90   
2        8634882   9.0                90   
3        8635215   9.0                90   
4        8642405  10.0                90   

                                                 UID  
0  10000_111.0_150.0_mastercard_117.0_debit_184.0...  
1          10003_555.0_128.0_visa_226.0_debit_NaN_90  
2          10003_555.0_128.0_visa_226.0_debit_NaN_90  
3          10003_555.0_128.0_visa_226.0_debit_NaN_90  
4          10003_555.0_128.0_visa_226.0_debit_NaN_90  


In [4]:
display(df_sequenced)

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V333,V334,V335,V336,V337,V338,V339,TransactionDay,Account_Start_Day,UID
0,3169988,0,4050851,29.000,W,10000,111.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.0,-37,10000_111.0_150.0_mastercard_117.0_debit_184.0...
1,3328484,0,8421815,39.394,C,10003,555.0,128.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,97.0,90,10003_555.0_128.0_visa_226.0_debit_NaN_90
2,3337343,0,8634882,10.755,C,10003,555.0,128.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,99.0,90,10003_555.0_128.0_visa_226.0_debit_NaN_90
3,3337365,0,8635215,19.093,C,10003,555.0,128.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,99.0,90,10003_555.0_128.0_visa_226.0_debit_NaN_90
4,3337822,0,8642405,19.093,C,10003,555.0,128.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,90,10003_555.0_128.0_visa_226.0_debit_NaN_90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3520757,0,14063430,25.000,S,9999,174.0,150.0,visa,226.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,162.0,35,9999_174.0_150.0_visa_226.0_debit_330.0_35
590536,3529705,0,14320277,20.000,S,9999,174.0,150.0,visa,226.0,...,0.0,0.0,40.0,0.0,0.0,0.0,0.0,165.0,35,9999_174.0_150.0_visa_226.0_debit_330.0_35
590537,3532261,0,14398268,20.000,S,9999,174.0,150.0,visa,226.0,...,20.0,20.0,60.0,20.0,0.0,0.0,0.0,166.0,35,9999_174.0_150.0_visa_226.0_debit_330.0_35
590538,3535099,0,14482385,30.000,S,9999,174.0,150.0,visa,226.0,...,40.0,20.0,80.0,40.0,0.0,0.0,0.0,167.0,35,9999_174.0_150.0_visa_226.0_debit_330.0_35


In [5]:
transactions_per_uid = df_sequenced['UID'].value_counts()
print('Credit cards with 1 transaction')
print(len(transactions_per_uid[transactions_per_uid == 1]))
print('Credit cards with 2 transaction')
print(len(transactions_per_uid[transactions_per_uid == 2]))
print('Credit cards with 3 transaction')
print(len(transactions_per_uid[transactions_per_uid == 3]))
print('Credit cards with 4 transaction')
print(len(transactions_per_uid[transactions_per_uid == 4]))
print('Credit cards with 5 transaction')
print(len(transactions_per_uid[transactions_per_uid == 5]))

Credit cards with 1 transaction
128982
Credit cards with 2 transaction
36933
Credit cards with 3 transaction
17227
Credit cards with 4 transaction
10137
Credit cards with 5 transaction
6784


In [6]:
fraud_by_uid = df_sequenced.groupby('UID')['isFraud'].agg(['count', 'sum', 'mean'])

fraud_by_uid = fraud_by_uid.rename(columns={
    'count': 'Total_Transactions',
    'sum': 'Fraudulent_Transactions',
    'mean': 'Fraud_Ratio'
})

# display(fraud_by_uid.sort_values(by='Total_Transactions', ascending=False).head(100))
display(fraud_by_uid[(fraud_by_uid['Total_Transactions'] == 1) & (fraud_by_uid['Fraudulent_Transactions'] > 0)])
display(fraud_by_uid[(fraud_by_uid['Total_Transactions'] == 10) & (fraud_by_uid['Fraudulent_Transactions'] > 0)])

,Total_Transactions,Fraudulent_Transactions,Fraud_Ratio
UID,,,
10004_529.0_150.0_visa_162.0_credit_299.0_35,1,1,1.0
10011_NaN_NaN_NaN_NaN_NaN_476.0_50,1,1,1.0
10019_590.0_150.0_mastercard_166.0_debit_264.0_71,1,1,1.0
10023_111.0_150.0_visa_226.0_debit_204.0_7,1,1,1.0
10023_111.0_150.0_visa_226.0_debit_325.0_-167,1,1,1.0
...,...,...,...
9941_273.0_185.0_mastercard_224.0_credit_NaN_180,1,1,1.0
9977_555.0_150.0_mastercard_203.0_credit_NaN_101,1,1,1.0
9992_455.0_150.0_mastercard_126.0_debit_264.0_94,1,1,1.0


,Total_Transactions,Fraudulent_Transactions,Fraud_Ratio
UID,,,
10023_111.0_150.0_visa_226.0_debit_325.0_69,10,1,0.1
10568_204.0_185.0_visa_226.0_credit_NaN_14,10,1,0.1
10568_204.0_185.0_visa_226.0_credit_NaN_25,10,5,0.5
10568_204.0_185.0_visa_226.0_credit_NaN_53,10,3,0.3
10568_204.0_185.0_visa_226.0_credit_NaN_62,10,9,0.9
...,...,...,...
9633_130.0_185.0_visa_138.0_debit_NaN_138,10,1,0.1
9633_130.0_185.0_visa_138.0_debit_NaN_151,10,7,0.7
9633_130.0_185.0_visa_138.0_debit_NaN_164,10,3,0.3


In [18]:
# Find the exact timestamp of the FIRST transaction for every individual UID
df_sequenced['UID_First_TransactionDT'] = df_sequenced.groupby('UID')['TransactionDT'].transform('min')

# Subtract the first timestamp from the current row's timestamp
df_sequenced['Time_Since_First_Observed_Txn'] = df_sequenced['TransactionDT'] - df_sequenced['UID_First_TransactionDT']

df_sequenced['Days_Since_First_Observed_Txn'] = df_sequenced['Time_Since_First_Observed_Txn'] / (24 * 60 * 60)

# Calculate the Hour of the Day (0 to 23)
# Divide total seconds by 3600 to get total hours, then modulo 24 to get the specific hour of the day.
df_sequenced['Hour_of_Day'] = np.floor(df_sequenced['TransactionDT'] / 3600) % 24

# Calculate the Day of the Week (0 to 6)
# Divide total seconds by 86400 (seconds in a day) to get total days, then modulo 7 to get the day of the week.
df_sequenced['Day_of_Week'] = np.floor(df_sequenced['TransactionDT'] / (24 * 60 * 60)) % 7

In [8]:
# 1. Define the irrelevant columns you want to discard
features_to_drop = [
    'TransactionDT',           # Replaced by our normalized time features
    'TransactionDay',          # Intermediate calculation step
    'UID_First_TransactionDT', # Intermediate calculation step
]

# Drop those columns (errors='ignore' prevents crashes if a column is already gone)
df_reduced = df_sequenced.drop(columns=features_to_drop, errors='ignore')

# 2. Fill missing values instead of dropping rows
# -999 is a standard placeholder for neural networks to signify a missing numerical value.
# For categorical string columns, you might want to fill with 'Unknown' beforehand.
df_clean = df_reduced.fillna(-999)

# 3. Group by UID and convert to nested lists of dictionaries
# We drop 'UID' from the lambda function so it doesn't repeat inside every timestep
user_sequences = df_clean.groupby('UID').apply(
    lambda x: x.drop(columns=['UID']).to_dict('records')
)

# 4. Export to a JSONL file
output_file = './data/ieee_fraud_sequences_full_embeddings.jsonl'

with open(output_file, 'w') as f:
    # .items() gives us the UID (the index) and the sequence (the list of dicts)
    for uid, sequence in user_sequences.items():
        
        # Structure it perfectly: UID at the root, transactions nested inside
        user_json = {
            "UID": uid, 
            "transactions": sequence
        }
        
        f.write(json.dumps(user_json) + '\n')

print(f"Successfully saved {len(user_sequences)} full-feature sequences to {output_file}")

/var/folders/y_/yz29kmqj7c71jwl9f5qf1r940000gn/T/ipykernel_39315/939667704.py:18: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_sequences = df_clean.groupby('UID').apply(


Successfully saved 222493 full-feature sequences to ./data/ieee_fraud_sequences_full_embeddings.jsonl


In [20]:
first_two_users = []

with open("./data/ieee_fraud_sequences_full_embeddings.jsonl", 'r') as f:
    for i, line in enumerate(f):
        if i >= 2:
            break
            
        user_data = json.loads(line)
        first_two_users.append(user_data)

with open("./data/full_embeddings_sample.jsonl", "w") as f:
    for user in first_two_users:
        f.write(json.dumps(user) + '\n')